# TTC gold layer

This notebook builds gold-layer insight tables from the silver TTC streaming datasets.

Gold outputs created in `dev.gold`:


These tables are designed for dashboards, operational monitoring, and downstream analytics.

## Gold layer design assumptions

This build uses PySpark plus Spark SQL DDL to materialize a Power BI-friendly star schema in `dev.gold`.

Key assumptions used in the implementation:

* Scheduled stop-level arrival times come from `sql_server_external.dbo.stop_times` because the Silver trip update feed does not contain timetable baselines.
* `dev.gold.dim_routes` is created as a supporting conformed dimension because the fact table requires a route foreign key.
* `dev.gold.gold_dim_weather_performance` is filtered to Toronto weather because TTC operations are Toronto-specific.
* Late-arriving Silver records are reconciled with latest-record deduplication by `ingested_at`; an optional watermark template is included at the end for converting this notebook into an incremental streaming pipeline.
* Power BI guidance:
  * `dev.gold.gold_active_fleet_status` is tuned for DirectQuery live visuals.
  * `dev.gold.gold_fact_transit_performance`, `dev.gold.gold_dim_alert_impact`, and `dev.gold.gold_dim_weather_performance` are compacted and analyzed for Import or hybrid semantic models.


In [0]:
from pyspark.sql import functions as F, Window

SOURCE_TABLES = {
    "trip_updates": "dev.silver.ttc_trip_updates_silver",
    "vehicle_positions": "dev.silver.ttc_vehicle_positions_silver",
    "alerts": "dev.silver.ttc_alerts_silver",
    "weather": "dev.silver.weather_silver",
    "stop_times": "sql_server_external.dbo.stop_times",
}

GOLD_SCHEMA = "dev.gold"
TORONTO_TZ = "America/Toronto"
ALERT_DEFAULT_DURATION_HOURS = 4
SNAPSHOT_LOOKBACK_HOURS = 2
BUNCHING_DISTANCE_METERS = 500.0
BUNCHING_TIME_MINUTES = 3.0

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.cbo.enabled", "true")


def write_gold_table(df, table_name, zorder_cols=None):
    temp_view = table_name.replace(".", "_").replace("-", "_") + "__tmp"
    df.createOrReplaceTempView(temp_view)
    spark.sql(f"""
        CREATE OR REPLACE TABLE {table_name}
        USING DELTA
        TBLPROPERTIES (
          'delta.autoOptimize.optimizeWrite' = 'true',
          'delta.autoOptimize.autoCompact' = 'true'
        )
        AS SELECT * FROM {temp_view}
    """)
    spark.sql(f"ANALYZE TABLE {table_name} COMPUTE STATISTICS FOR ALL COLUMNS")
    if zorder_cols:
        spark.sql(f"OPTIMIZE {table_name} ZORDER BY ({', '.join(zorder_cols)})")
    else:
        spark.sql(f"OPTIMIZE {table_name}")


def haversine_m(lat1, lon1, lat2, lon2):
    lat1_rad = F.radians(lat1)
    lon1_rad = F.radians(lon1)
    lat2_rad = F.radians(lat2)
    lon2_rad = F.radians(lon2)
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = F.pow(F.sin(dlat / 2), 2) + F.cos(lat1_rad) * F.cos(lat2_rad) * F.pow(F.sin(dlon / 2), 2)
    return F.lit(6371000.0) * 2 * F.asin(F.sqrt(a))


In [0]:
trip_schedule = (
    trip_updates.alias("tu")
    .join(
        stop_times.alias("st"),
        on=["trip_id", "stop_id", "stop_sequence"],
        how="left",
    )
    .withColumn("scheduled_service_date_local", F.expr("date_add(service_date_local, arrival_day_offset)"))
    .withColumn(
        "scheduled_arrival_ts_local",
        F.to_timestamp(
            F.concat_ws(
                " ",
                F.date_format("scheduled_service_date_local", "yyyy-MM-dd"),
                F.format_string("%02d:%02d:%02d", F.col("arrival_hour_mod"), F.col("arrival_minute"), F.col("arrival_second")),
            )
        ),
    )
    .withColumn("scheduled_arrival_ts_utc", F.to_utc_timestamp("scheduled_arrival_ts_local", TORONTO_TZ))
    .withColumn(
        "schedule_variance_minutes",
        F.round((F.col("actual_arrival_ts_utc").cast("long") - F.col("scheduled_arrival_ts_utc").cast("long")) / F.lit(60.0), 2),
    )
    .withColumn(
        "delay_minutes",
        F.round(F.greatest(F.col("schedule_variance_minutes"), F.lit(0.0)), 2),
    )
    .withColumn(
        "otp_status",
        F.when(F.col("scheduled_arrival_ts_utc").isNull(), F.lit("NO_SCHEDULE_MATCH"))
         .when((F.col("schedule_variance_minutes") >= F.lit(-1.0)) & (F.col("schedule_variance_minutes") <= F.lit(5.0)), F.lit("ON_TIME"))
         .when(F.col("schedule_variance_minutes") < F.lit(-1.0), F.lit("EARLY"))
         .otherwise(F.lit("LATE")),
    )
    .withColumn(
        "otp_flag",
        F.when((F.col("schedule_variance_minutes") >= F.lit(-1.0)) & (F.col("schedule_variance_minutes") <= F.lit(5.0)), F.lit(1)).otherwise(F.lit(0)),
    )
)

alerts_for_join = alerts_detailed.select(
    F.col("alert_sk"),
    F.col("alert_business_id").alias("alert_id"),
    F.col("route_id").alias("alert_route_id"),
    F.col("stop_id").alias("alert_stop_id"),
    F.col("alert_start_ts_utc"),
    F.col("alert_end_ts_utc"),
    F.col("cause_category").alias("alert_cause_category"),
    F.col("effect").alias("alert_effect"),
)

fact_with_alerts = (
    trip_schedule.alias("f")
    .join(
        alerts_for_join.alias("a"),
        on=(F.col("f.route_id") == F.col("a.alert_route_id"))
           & ((F.col("f.stop_id") == F.col("a.alert_stop_id")) | F.col("a.alert_stop_id").isNull())
           & (F.col("f.actual_arrival_ts_utc") >= F.col("a.alert_start_ts_utc"))
           & (F.col("f.actual_arrival_ts_utc") <= F.col("a.alert_end_ts_utc")),
        how="left",
    )
    .withColumn(
        "alert_match_rn",
        F.row_number().over(
            Window.partitionBy("fact_nk")
            .orderBy(
                F.when(F.col("f.stop_id") == F.col("a.alert_stop_id"), F.lit(0)).otherwise(F.lit(1)),
                F.col("a.alert_start_ts_utc").desc_nulls_last(),
            )
        ),
    )
    .filter((F.col("alert_match_rn") == 1) | F.col("a.alert_id").isNull())
    .drop("alert_match_rn", "alert_route_id", "alert_stop_id")
)

fact_transit_performance = (
    fact_with_alerts.alias("f")
    .join(
        weather_hourly.select(
            "weather_sk",
            "weather_hour_local",
            "condition_text",
            "temperature_c",
            "precipitation_mm",
            "wind_kph",
            "humidity",
        ).alias("w"),
        on=(F.col("f.service_hour_local") == F.col("w.weather_hour_local")) & (F.col("f.weather_sk") == F.col("w.weather_sk")),
        how="left",
    )
    .select(
        "fact_transit_performance_sk",
        "fact_nk",
        "route_sk",
        "weather_sk",
        "alert_sk",
        "trip_id",
        "vehicle_id",
        "route_id",
        "stop_id",
        "stop_sequence",
        "schedule_relationship",
        F.col("actual_arrival_ts_utc").alias("actual_arrival_ts_utc"),
        F.col("scheduled_arrival_ts_utc").alias("scheduled_arrival_ts_utc"),
        "event_timestamp",
        "service_date_local",
        "service_hour_local",
        "schedule_variance_minutes",
        "delay_minutes",
        "otp_status",
        "otp_flag",
        F.when(F.col("scheduled_arrival_ts_utc").isNotNull(), F.lit(1)).otherwise(F.lit(0)).alias("schedule_match_flag"),
        F.col("w.condition_text").alias("weather_condition_text"),
        F.col("w.temperature_c").alias("weather_temperature_c"),
        F.col("w.precipitation_mm").alias("weather_precipitation_mm"),
        F.col("w.wind_kph").alias("weather_wind_kph"),
        F.col("w.humidity").alias("weather_humidity"),
        "alert_id",
        "alert_cause_category",
        "alert_effect",
        "ingested_at",
    )
)

write_gold_table(
    fact_transit_performance,
    f"{GOLD_SCHEMA}.gold_fact_transit_performance",
    zorder_cols=["route_id", "event_timestamp"],
)

In [0]:
vehicle_positions_recent = (
    spark.table(SOURCE_TABLES["vehicle_positions"])
    .filter(
        F.col("route_id").isNotNull()
        & F.col("vehicle_id").isNotNull()
        & F.col("event_timestamp").isNotNull()
        & (F.col("event_timestamp") >= F.current_timestamp() - F.expr(f"INTERVAL {SNAPSHOT_LOOKBACK_HOURS} HOURS"))
    )
    .withColumn(
        "latest_vehicle_rn",
        F.row_number().over(
            Window.partitionBy("vehicle_id")
            .orderBy(F.col("event_timestamp").desc_nulls_last(), F.col("ingested_at").desc_nulls_last())
        ),
    )
    .filter(F.col("latest_vehicle_rn") == 1)
    .drop("latest_vehicle_rn")
    .withColumn("speed_kph", F.round(F.col("speed") * F.lit(3.6), 2))
    .withColumn("snapshot_hour_local", F.date_trunc("hour", F.from_utc_timestamp("event_timestamp", TORONTO_TZ)))
    .withColumn("route_sk", F.abs(F.xxhash64("route_id")))
)

route_schedule_snapshot = (
    trip_updates
    .filter(F.col("event_timestamp") >= F.current_timestamp() - F.expr(f"INTERVAL {SNAPSHOT_LOOKBACK_HOURS} HOURS"))
    .groupBy("route_id")
    .agg(
        F.countDistinct("trip_id").alias("scheduled_trip_count"),
        F.countDistinct("vehicle_id").alias("scheduled_vehicle_count"),
    )
)

route_order_window = Window.partitionBy("route_id").orderBy(F.col("current_stop_sequence").asc_nulls_last(), F.col("event_timestamp").asc_nulls_last(), F.col("vehicle_id"))
vehicle_pairs = (
    vehicle_positions_recent
    .withColumn("prev_latitude", F.lag("latitude").over(route_order_window))
    .withColumn("prev_longitude", F.lag("longitude").over(route_order_window))
    .withColumn("prev_event_timestamp", F.lag("event_timestamp").over(route_order_window))
    .withColumn(
        "distance_to_prev_vehicle_m",
        haversine_m("latitude", "longitude", "prev_latitude", "prev_longitude"),
    )
    .withColumn(
        "minutes_to_prev_vehicle",
        F.abs(F.col("event_timestamp").cast("long") - F.col("prev_event_timestamp").cast("long")) / F.lit(60.0),
    )
    .withColumn(
        "bunched_vehicle_flag",
        F.when(
            (F.col("distance_to_prev_vehicle_m") <= F.lit(BUNCHING_DISTANCE_METERS))
            & (F.col("minutes_to_prev_vehicle") <= F.lit(BUNCHING_TIME_MINUTES)),
            F.lit(1),
        ).otherwise(F.lit(0)),
    )
)

route_fleet_metrics = (
    vehicle_pairs
    .groupBy("route_id")
    .agg(
        F.countDistinct("vehicle_id").alias("active_vehicle_count"),
        F.round(F.avg("speed_kph"), 2).alias("avg_fleet_speed_kph"),
        F.round(F.avg(F.col("bunched_vehicle_flag").cast("double")), 4).alias("vehicle_bunching_index"),
        F.max("event_timestamp").alias("snapshot_ts_utc"),
    )
    .join(route_schedule_snapshot, on="route_id", how="left")
    .withColumn(
        "active_vs_scheduled_ratio",
        F.round(
            F.when(F.col("scheduled_vehicle_count").isNull() | (F.col("scheduled_vehicle_count") == 0), None)
             .otherwise(F.col("active_vehicle_count") / F.col("scheduled_vehicle_count")),
            4,
        ),
    )
)

gold_active_fleet_status = (
    vehicle_pairs.alias("v")
    .join(route_fleet_metrics.alias("m"), on="route_id", how="left")
    .select(
        F.abs(F.xxhash64("v.vehicle_id", F.date_format("v.event_timestamp", "yyyy-MM-dd HH:mm:ss"))).alias("fleet_status_sk"),
        "v.route_sk",
        "v.vehicle_id",
        "v.trip_id",
        "v.route_id",
        "v.stop_id",
        "v.current_stop_sequence",
        "v.current_status",
        "v.latitude",
        "v.longitude",
        "v.bearing",
        "v.speed",
        "v.speed_kph",
        "v.event_timestamp",
        "v.ingested_at",
        "v.snapshot_hour_local",
        "m.snapshot_ts_utc",
        "m.active_vehicle_count",
        "m.scheduled_trip_count",
        "m.scheduled_vehicle_count",
        "m.active_vs_scheduled_ratio",
        "m.vehicle_bunching_index",
        "m.avg_fleet_speed_kph",
        "v.distance_to_prev_vehicle_m",
        "v.minutes_to_prev_vehicle",
        "v.bunched_vehicle_flag",
    )
)

write_gold_table(
    gold_active_fleet_status,
    f"{GOLD_SCHEMA}.gold_active_fleet_status",
    zorder_cols=["route_id", "vehicle_id", "event_timestamp"],
)

In [0]:
fact_df = spark.table(f"{GOLD_SCHEMA}.gold_fact_transit_performance")
vehicle_positions_for_weather = (
    spark.table(SOURCE_TABLES["vehicle_positions"])
    .filter(F.col("event_timestamp").isNotNull())
    .withColumn("service_hour_local", F.date_trunc("hour", F.from_utc_timestamp("event_timestamp", TORONTO_TZ)))
    .withColumn("speed_kph", F.col("speed") * F.lit(3.6))
)

hourly_delay_metrics = (
    fact_df
    .groupBy("service_hour_local")
    .agg(
        F.count("*").alias("trip_stop_arrival_count"),
        F.round(F.avg("delay_minutes"), 2).alias("avg_delay_minutes"),
        F.round(F.avg(F.col("otp_flag").cast("double")), 4).alias("otp_rate"),
        F.round(F.avg("schedule_variance_minutes"), 2).alias("avg_schedule_variance_minutes"),
    )
)

hourly_speed_metrics = (
    vehicle_positions_for_weather
    .groupBy("service_hour_local")
    .agg(F.round(F.avg("speed_kph"), 2).alias("avg_fleet_speed_kph"))
)

weather_performance = (
    weather_hourly.alias("w")
    .join(hourly_delay_metrics.alias("d"), F.col("w.weather_hour_local") == F.col("d.service_hour_local"), "left")
    .join(hourly_speed_metrics.alias("s"), F.col("w.weather_hour_local") == F.col("s.service_hour_local"), "left")
    .select(
        "w.weather_sk",
        F.to_date("w.weather_hour_local").alias("service_date_local"),
        F.hour("w.weather_hour_local").alias("service_hour_of_day"),
        "w.weather_hour_local",
        "w.city",
        "w.condition_text",
        "w.temperature_c",
        "w.precipitation_mm",
        "w.wind_kph",
        "w.humidity",
        "w.cloud",
        "w.air_quality_pm2_5",
        F.col("d.trip_stop_arrival_count"),
        F.col("d.avg_delay_minutes"),
        F.col("d.otp_rate"),
        F.col("d.avg_schedule_variance_minutes"),
        F.col("s.avg_fleet_speed_kph"),
    )
)

baseline_weather = (
    weather_performance
    .filter(
        (F.coalesce(F.col("precipitation_mm"), F.lit(0.0)) == F.lit(0.0))
        & (~F.lower(F.coalesce(F.col("condition_text"), F.lit(""))).rlike("rain|snow|storm|sleet|freezing|ice"))
    )
    .agg(
        F.avg("avg_delay_minutes").alias("baseline_delay_minutes"),
        F.avg("avg_fleet_speed_kph").alias("baseline_speed_kph"),
    )
    .first()
)

baseline_delay_minutes = float((baseline_weather["baseline_delay_minutes"] if baseline_weather else None) or 1.0)
baseline_speed_kph = float((baseline_weather["baseline_speed_kph"] if baseline_weather else None) or 1.0)

weather_performance_final = (
    weather_performance
    .withColumn("baseline_delay_minutes", F.lit(baseline_delay_minutes))
    .withColumn("baseline_speed_kph", F.lit(baseline_speed_kph))
    .withColumn(
        "weather_delay_factor",
        F.round(
            F.when(F.col("baseline_delay_minutes") == 0, None)
             .otherwise(F.col("avg_delay_minutes") / F.col("baseline_delay_minutes")),
            4,
        ),
    )
    .withColumn(
        "speed_drop_pct",
        F.round(
            F.when(F.col("baseline_speed_kph") == 0, None)
             .otherwise(((F.col("baseline_speed_kph") - F.col("avg_fleet_speed_kph")) / F.col("baseline_speed_kph")) * F.lit(100.0)),
            2,
        ),
    )
)

write_gold_table(
    weather_performance_final,
    f"{GOLD_SCHEMA}.gold_dim_weather_performance",
    zorder_cols=["weather_hour_local"],
)

alert_fact_impact = (
    fact_df
    .filter(F.col("alert_sk").isNotNull())
    .groupBy("alert_sk")
    .agg(
        F.countDistinct("trip_id").alias("impacted_trip_count"),
        F.countDistinct(F.when(F.col("delay_minutes") > F.lit(0), F.col("trip_id"))).alias("delayed_trip_count"),
        F.countDistinct(F.when(F.col("schedule_relationship") != F.lit("SCHEDULED"), F.col("trip_id"))).alias("modified_or_canceled_trip_count"),
        F.round(F.avg("delay_minutes"), 2).alias("avg_delay_minutes_per_alert"),
    )
)

alert_impact = (
    alert_incidents
    .join(alert_fact_impact, on="alert_sk", how="left")
    .withColumn("alert_duration_minutes", F.round((F.col("alert_end_ts_utc").cast("long") - F.col("alert_start_ts_utc").cast("long")) / F.lit(60.0), 2))
    .withColumn(
        "incident_frequency_by_type",
        F.count("*").over(Window.partitionBy("cause_category")),
    )
    .withColumn(
        "impact_score",
        F.coalesce(F.col("delayed_trip_count"), F.lit(0)) + F.coalesce(F.col("modified_or_canceled_trip_count"), F.lit(0)),
    )
    .select(
        "alert_sk",
        "alert_id",
        F.abs(F.xxhash64("route_id")).alias("route_sk"),
        "route_id",
        "cause",
        "cause_category",
        "effect",
        "message",
        "alert_start_ts_utc",
        "alert_end_ts_utc",
        "alert_duration_minutes",
        "impacted_stop_count",
        "impacted_stop_ids",
        "incident_frequency_by_type",
        F.coalesce(F.col("impacted_trip_count"), F.lit(0)).alias("impacted_trip_count"),
        F.coalesce(F.col("delayed_trip_count"), F.lit(0)).alias("delayed_trip_count"),
        F.coalesce(F.col("modified_or_canceled_trip_count"), F.lit(0)).alias("modified_or_canceled_trip_count"),
        F.coalesce(F.col("avg_delay_minutes_per_alert"), F.lit(0.0)).alias("avg_delay_minutes_per_alert"),
        "impact_score",
    )
)

write_gold_table(
    alert_impact,
    f"{GOLD_SCHEMA}.gold_dim_alert_impact",
    zorder_cols=["route_id", "alert_start_ts_utc"],
)